# Fake vs real titles — DistilBERT demo

Walk this notebook top to bottom in an interview. Training lives in `python -m src.train`; this file is the story plus a fast baseline.

Labels: **0 = real**, **1 = fake**.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import load_titles, make_splits

df = load_titles()
print(df.shape, df["label"].value_counts().to_dict())
df.groupby("label").head(2)

## Baseline: TF-IDF + logistic regression

Same split as DistilBERT. If this already scores very high, the titles leak **style** (news desk vs clickbait). That is expected on the bundled CSV.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline

splits = make_splits(df)
clf = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20_000)),
    ("lr", LogisticRegression(max_iter=1000, random_state=42)),
])
clf.fit(splits.train["title"], splits.train["label"])
pred = clf.predict(splits.test["title"])
print(classification_report(splits.test["label"], pred, target_names=["real", "fake"], digits=4))
print(confusion_matrix(splits.test["label"], pred))

## DistilBERT

From the repo root, once:

```bash
python -m src.train
python -m src.evaluate
```

Then run the cell below. It loads `checkpoints/best_model/` if you have trained.

In [ ]:
from src.config import BEST_MODEL_DIR

examples = [
    "The Federal Reserve holds interest rates after a two-day meeting",
    "WHO reports a decline in measles cases in East Africa",
    "You won't believe this spice reverses aging overnight",
    "Secret documents prove the moon landing was filmed in a hangar",
]

if not BEST_MODEL_DIR.exists():
    print("No checkpoint yet. From the repo root run: python -m src.train")
else:
    from src.predict import load_model, predict_one
    tok, model = load_model()
    for text in examples:
        label, p_real, p_fake = predict_one(text, tok, model)
        print(f"{label:4s}  P(real)={p_real:.3f}  P(fake)={p_fake:.3f}  |  {text}")

## Lines to remember

1. This is title classification, not a fact checker.
2. DistilBERT = smaller BERT, same fine-tuning recipe.
3. Original Fakeddit run: **79.6%** accuracy with `bert-base-uncased` on 1,000 test titles.
4. Custom BERT-from-scratch and Captum were dropped so the live demo stays explainable.

Full walkthrough: `DESIGN_DOC.md`.